# ARIS: End-to-End Tutorial and Workflow
### Differentiable Analysis-by-Synthesis for Voice Research

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/N1r/ARIS_nsf/blob/main/notebooks/ARIS_Tutorial_and_Workflow.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-ARIS__nsf-181717.svg)](https://github.com/N1r/ARIS_nsf)

This notebook provides a self-contained walkthrough of ARIS (Analytic Resonance for Interpretable Synthesis). In Colab, select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Package installation and dependency resolution use **uv** throughout.

1. **Environment Setup & Diagnostics**: verifying GPU and dependencies
2. **Audio Inspection & F0 Tracking**: acoustic feature extraction using WORLD
3. **Dataset Manifest Preparation**: normalization and train/val/test splitting
4. **Model Training**: training the differentiable `aria-golf` neural source-filter vocoder
5. **Acoustic Manipulation & Resynthesis**: targeted formant ($F_1$), pitch ($F_0$), and glottal source ($R_d$) modifications
6. **Interactive Studio**: launching the browser-based workspace for continuous parameter exploration
7. **Export**: packaging stimuli together with provenance metadata


## Scope and scientific interpretation

ARIS controls parameters inside a trained analysis-by-synthesis model. Those controls are not direct physiological measurements or perceptual labels. Treat the generated WAV files as experimental materials that still require independent acoustic validation. In particular:

- compare every manipulation with both the original recording and an unmanipulated reconstruction;
- remeasure the intended outcome (for example F0, F1/F2, H1–H2, CPP, HNR, or waveform amplitude) in the output;
- inspect intelligibility, artifacts, clipping, and unintended covariation before a perception study;
- archive the checkpoint hash, dataset fingerprint, control metadata, and analysis code.

This tutorial demonstrates whole-utterance F0, formant, and source controls. ARIS does not currently manipulate duration, speech rate, or a selected interval within an utterance.


## 1. Setup and Environment

Verify GPU availability, clone the repository when running on Colab, install uv with its official installer, and use uv to install the lockfile-resolved dependencies into the active notebook kernel. This avoids a runtime restart.


In [ ]:
import shutil
import subprocess
import tempfile

print("[0/7] Hardware check", flush=True)
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    subprocess.run([nvidia_smi, "--query-gpu=name,memory.total,driver_version",
                    "--format=csv,noheader"], check=True)
else:
    print("No NVIDIA GPU detected. Inference can use CPU; training will be slow.")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def run(*args):
    return subprocess.run(args, check=True)

print("\n[1/7] Environment setup with uv", flush=True)
runtime_tmp = Path("/tmp/aris-notebook")
runtime_tmp.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(runtime_tmp)
tempfile.tempdir = str(runtime_tmp)
try:
    import google.colab  # type: ignore[import-not-found]
    in_colab = True
except ImportError:
    in_colab = False
print(f"Runtime: {'Google Colab' if in_colab else 'local Jupyter'}")

# Install uv only when the current runtime does not already provide it.
uv = shutil.which("uv")
if uv is None:
    run("bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"{Path.home() / '.local' / 'bin'}:{os.environ['PATH']}"
    uv = shutil.which("uv")
assert uv is not None, "uv installation did not produce an executable"
run(uv, "--version")

# Find a local checkout or clone once when running in Colab.
if not Path("pyproject.toml").is_file() and Path("../pyproject.toml").is_file():
    os.chdir("..")
elif not Path("pyproject.toml").is_file():
    checkout = Path("ARIS_nsf")
    if not checkout.exists():
        run("git", "clone", "--depth", "1", "https://github.com/N1r/ARIS_nsf.git")
    os.chdir(checkout)

if in_colab:
    # Colab's running kernel cannot switch to .venv without a restart. Export the lockfile
    # and let uv install those exact versions into the active kernel instead.
    requirements = "/tmp/aris-requirements.txt"
    run(uv, "export", "--quiet", "--frozen", "--all-extras", "--no-dev",
        "--no-emit-project", "--no-hashes", "--format", "requirements-txt",
        "--output-file", requirements)
    run(uv, "pip", "install", "--system", "--requirement", requirements)
    run(uv, "pip", "install", "--system", "--no-deps", "--editable", ".")
else:
    # Local notebooks use the project environment. Launch Jupyter with:
    # uv run --with jupyter jupyter lab
    sync_env = os.environ.copy()
    sync_env.pop("VIRTUAL_ENV", None)
    subprocess.run([uv, "sync", "--locked", "--all-extras"], check=True, env=sync_env)

print(f"Python: {sys.executable}")
print(f"Project: {Path.cwd()}")
print("Environment setup complete.")


In [ ]:
import aris

print("\n[1/7] Dependency diagnostics")
findings = aris.doctor()
for item in findings:
    status = "OK" if item.ok else "--"
    print(f"  [{status:2}] {item.name:12} {item.detail}")
required_failures = [item for item in findings if not item.ok and item.name != "ffmpeg"]
if required_failures:
    raise RuntimeError("Required environment checks failed: " +
                       ", ".join(item.name for item in required_failures))
print(f"ARIS: {aris.__file__}")
print("Dependency diagnostics complete.")


## 2. Audio Data and Acoustic Features

Download the reference Mandarin female speaker package (`demo_f024`, 16 kHz). It contains audio, a prepared dataset, and a pretrained checkpoint, so the inference section remains useful even if you skip or interrupt the demonstration training. Then inspect one utterance and extract its fundamental-frequency ($F_0$) trajectory with WORLD.


In [ ]:
import urllib.request
import zipfile
from pathlib import Path

print("\n[2/7] Reference data")
# Fetch reference demo package if not present.
archive_path = Path("aris_f024_demo.zip")
if not Path("demo_f024").exists():
    url = "https://github.com/N1r/ARIS_nsf/releases/download/v0.1.0/aris_f024_demo.zip"
    urllib.request.urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(".")
    print(f"Downloaded and extracted: {archive_path}")
else:
    print("Using cached demo_f024 directory.")
audio_files = sorted(Path("demo_f024/dataset/audio").glob("*.wav"))
checkpoint_path = Path("demo_f024/experiment/runs/checkpoints/last.ckpt")
if not audio_files or not checkpoint_path.is_file():
    raise FileNotFoundError("Demo package is incomplete: expected WAV files and last.ckpt")
print(f"Reference package ready: {len(audio_files)} WAV files; checkpoint={checkpoint_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from aris.audio import estimate_f0, read_audio

# Inspect sample utterance
sample_path = audio_files[0]
audio, sr = read_audio(sample_path)
f0, backend = estimate_f0(audio, sr, floor_hz=50, ceiling_hz=800, method="pyworld")
time_axis = np.linspace(0, len(audio) / sr, len(f0))

voiced_f0 = f0[f0 > 0]
print(f"Sample: {sample_path.name}")
print(f"Sampling rate: {sr} Hz | duration: {len(audio)/sr:.2f} s | F0 backend: {backend}")
print(f"Median voiced F0: {np.median(voiced_f0):.1f} Hz | voiced frames: {len(voiced_f0)/len(f0):.1%}")
display(Audio(audio, rate=sr))

# Plot waveform and F0 trajectory
fig, (ax_wave, ax_f0) = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
ax_wave.plot(np.linspace(0, len(audio) / sr, len(audio)), audio, color="#333333", linewidth=0.8)
ax_wave.set_ylabel("Amplitude")
ax_wave.set_title("Waveform")

f0_voiced = np.where(f0 > 0, f0, np.nan)
ax_f0.plot(time_axis, f0_voiced, color="#1b7837", linewidth=2.0)
ax_f0.set_ylabel("F0 (Hz)")
ax_f0.set_xlabel("Time (s)")
ax_f0.set_ylim(50, 400)
ax_f0.set_title("Fundamental Frequency (F0)")
plt.tight_layout()
plt.show()


## 3. Dataset Preparation

Process raw audio into a standardized experiment directory: resample to 16 kHz, precompute $F_0$ features, and partition into train/validation/test splits.


In [ ]:
from collections import Counter

print("\n[3/7] Dataset preparation")
# Build manifest and extract acoustic features across the corpus.
manifest = aris.prepare(
    source="demo_f024/dataset/audio",
    output="data/prepared_corpus",
    sample_rate=16000,
    f0_method="pyworld",
    min_duration=0.5,
    overwrite=True,
)

split_counts = Counter(record.split for record in manifest.records)
total_duration = sum(record.duration_s for record in manifest.records)
clipped_inputs = sum(record.clipped_fraction > 0 for record in manifest.records)
print(f"Prepared {len(manifest.records)} utterances ({total_duration/60:.1f} min).")
print("Split counts: " + ", ".join(f"{name}={split_counts.get(name, 0)}"
                                      for name in ("train", "validation", "test")))
print(f"Dataset fingerprint: {manifest.fingerprint}")
print(f"Input recordings with at least one clipped sample: {clipped_inputs}")
errors = aris.validate("data/prepared_corpus")
if errors:
    raise RuntimeError("Dataset validation failed:\n" + "\n".join(errors))
else:
    print("Dataset validation passed.")


## 4. Model Training

Configure and train an `aria-golf` vocoder on the prepared dataset. `aria-golf` models source-filter interaction using differentiable time-varying LPC and glottal flow wavetables, enabling independent control over vocal-tract resonances and glottal-pulse parameters.

> The Colab tutorial uses 1,000 optimization steps. This is enough to show a meaningful loss trajectory and produce a model that can be compared audibly with the original, while keeping the exercise short; it is not a claim of convergence or research-ready quality. On an NVIDIA L4, this run takes roughly two minutes after data preparation; a Colab T4 will generally be slower. The bundled 40,000-step checkpoint remains available as a stable quality reference.


In [ ]:
import csv
import time

print("\n[4/7] Tutorial model training")
training_steps = 1_000
# Initialize experiment configuration.
exp_dir = aris.init_experiment(
    dataset="data/prepared_corpus",
    output="experiments/tutorial_run",
    model="aria-golf",
    batch_size=16,
    max_steps=training_steps,
    workers=0,
    overwrite=True,
)

print(f"Experiment: {exp_dir}")
print(f"Model: aria-golf | batch size: 16 | max steps: {training_steps:,}")
started = time.perf_counter()
aris.train(exp_dir)
elapsed_minutes = (time.perf_counter() - started) / 60
trained_checkpoint = Path(exp_dir) / "runs/checkpoints/last.ckpt"
if not trained_checkpoint.is_file():
    raise FileNotFoundError(f"Training finished without checkpoint: {trained_checkpoint}")
print(f"Training time: {elapsed_minutes:.1f} min")
print(f"Tutorial checkpoint: {trained_checkpoint}")


In [ ]:
# Read Lightning's CSV log and inspect the optimization trajectory.
metric_files = sorted((Path(exp_dir) / "runs/metrics").glob("version_*/metrics.csv"))
if not metric_files:
    raise FileNotFoundError("Training completed without metrics.csv")
with metric_files[-1].open(newline="") as stream:
    metric_rows = list(csv.DictReader(stream))
train_points = [(int(row["step"]), float(row["train_loss"]))
                for row in metric_rows if row.get("train_loss")]
val_points = [(int(row["step"]), float(row["val_loss"]))
              for row in metric_rows if row.get("val_loss")]
if not train_points:
    raise RuntimeError("metrics.csv contains no train_loss values")

train_steps, train_losses = map(np.asarray, zip(*train_points))
window = min(100, len(train_losses))
smoothed = np.convolve(train_losses, np.ones(window) / window, mode="valid")
smooth_steps = train_steps[window - 1:]
first_median = float(np.median(train_losses[:window]))
last_median = float(np.median(train_losses[-window:]))
print(f"Training-loss median: first {window} steps={first_median:.4f} | "
      f"last {window} steps={last_median:.4f}")
if val_points:
    print(f"Validation loss: first={val_points[0][1]:.4f} | last={val_points[-1][1]:.4f} "
          f"| evaluations={len(val_points)}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(smooth_steps, smoothed, label=f"train loss ({window}-step mean)", color="#1b7837")
if val_points:
    val_steps, val_losses = zip(*val_points)
    ax.plot(val_steps, val_losses, "o-", label="validation loss", color="#762a83", markersize=3)
ax.set(xlabel="Optimization step", ylabel="Multi-scale spectral loss", title="Training trajectory")
ax.set_yscale("log")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Resynthesis and Acoustic Manipulation

Use the newly trained 1,000-step tutorial checkpoint to resynthesize held-out audio and generate controlled stimuli. This makes the audible result and the plotted loss trajectory refer to the same model. Treat it as a teaching result, not research-ready synthesis. For a stable quality reference, replace `checkpoint` and `experiment` below with the bundled 40,000-step demo paths.

- `f1_scale`: frame-wise scaling of the model's first-formant trajectory
- `pitch_semitones`: uniform semitone shift applied to the voiced pitch contour
- `glottal_rd_scale`: scaling of the model's glottal-source $R_d$ parameter


In [ ]:
import json
from pathlib import Path

print("\n[5/7] Resynthesis and controlled variants")
experiment = Path(exp_dir)
checkpoint = trained_checkpoint
recon_dir = Path("out/reconstruction")
stimuli_dir = Path("out/stimuli")

# 1. Baseline Resynthesis (synthesizes held-out test utterances)
aris.synthesize(experiment, checkpoint, recon_dir, overwrite=True)

# 2. Targeted Acoustic Manipulation
aris.manipulate(
    experiment,
    checkpoint,
    stimuli_dir,
    variants=[
        "f1_up:f1_scale=1.2",
        "pitch_down:pitch_semitones=-4",
        "rd_high:glottal_rd_scale=1.6",
        "rd_low:glottal_rd_scale=0.6",
    ],
    overwrite=True,
)

reconstructed = sorted(recon_dir.glob("*.wav"))
audit = json.loads((stimuli_dir / "manipulation.json").read_text())
if not reconstructed or not audit["outputs"]:
    raise RuntimeError("Rendering completed without auditable WAV outputs")
print(f"Baseline reconstructions: {len(reconstructed)}")
print(f"Checkpoint SHA-256: {audit['checkpoint_sha256']}")
for variant in audit["outputs"]:
    render = variant["render_audit"]
    print(f"  {variant['name']:<12} files={render['files_written']:<3} "
          f"hard-clipped sample proportion={render['clipped_fraction']:.6f} "
          f"controls={variant['controls']}")


In [ ]:
# Select an item from the current held-out split; do not rely on a fixed filename.
test_record = next(record for record in manifest.records if record.split == "test")
orig_wav = Path("demo_f024/dataset/audio") / test_record.source_path
recon_wav = recon_dir / f"{test_record.id}.wav"
f1_wav = stimuli_dir / "f1_up" / recon_wav.name
pitch_wav = stimuli_dir / "pitch_down" / recon_wav.name
rd_high_wav = stimuli_dir / "rd_high" / recon_wav.name
rd_low_wav = stimuli_dir / "rd_low" / recon_wav.name
for wav_path in (orig_wav, recon_wav, f1_wav, pitch_wav, rd_high_wav, rd_low_wav):
    if not wav_path.is_file():
        raise FileNotFoundError(wav_path)

comparisons = {
    "Original recording": orig_wav,
    "Reconstructed baseline": recon_wav,
    "F1 shift (+20%)": f1_wav,
    "Pitch shift (-4 semitones)": pitch_wav,
    "Higher model Rd (× 1.6)": rd_high_wav,
    "Lower model Rd (× 0.6)": rd_low_wav,
}
for label, wav_path in comparisons.items():
    print(f"{label}: {wav_path}")
    display(Audio(filename=str(wav_path)))


## 6. Interactive Studio

Launch the browser-based workbench for continuous slider adjustment and A/B listening. In Colab, `share=True` creates a temporary public Gradio URL. `launch_studio` is `False` by default so **Run all** can finish and export the stimuli; set it to `True` and run this cell manually when needed. The Studio supports quality control but does not provide experimental randomization, blinding, or playback calibration.


In [ ]:
print("\n[6/7] Interactive Studio")
launch_studio = False
if launch_studio:
    aris.launch_studio(workspace=".", share=in_colab, open_browser=not in_colab)
else:
    print("Studio skipped during Run all. Set launch_studio=True and rerun this cell to open it.")


## 7. Export Stimuli

Package generated stimuli and computational provenance metadata (`manipulation.json`) for experimental deployment. In Colab, uncomment the final two lines to download the ZIP.


In [ ]:
print("\n[7/7] Export")
archive = Path(shutil.make_archive("aris_stimuli", "zip", "out/stimuli"))
print(f"Saved: {archive.resolve()}")
print(f"Archive size: {archive.stat().st_size / 1_048_576:.1f} MiB")
print("Includes manipulation.json and one _render.json per condition.")
# from google.colab import files
# files.download("aris_stimuli.zip")
